# Tier-1 Notebook: Wall-Clock (E1) + Selection-Rule Ablation (E2)

Two independent experiments in one notebook. Section A runs first (fast); Section B runs second (long).

**Section A — E1: Wall-clock at ρ=0.7.** Updates the Phase 5 wall-clock measurement (which was at ρ=0.5) to the new headline cap. Phase 5 reported 1.7–2.4% overhead; at ρ=0.7 the Pass-2 query work is `N/(1-0.7) ≈ 3.33N` vs `N/(1-0.5) = 2N`, so the overhead is expected to be 4–6%. The paper needs the actual number. 30 timed runs at 2K samples each, no FID eval. ~1.5 hours.

**Section B — E2: Selection-rule ablation at FID-50K.** RTR Phase 1 showed random ≥ margin at FID-10K. For the §4.2 ablation table, we need the equivalent comparison at FID-50K multi-seed. 24 configs total: 4 selection rules × 2 step counts × 3 seeds. The 6 random rows are pre-loaded from the main-table CSV; only 18 new FID-50K runs needed. ~12 hours.

**Total time: ~13 hours.** Fits in a single Colab Pro overnight session.

**Persistence.** Separate CSVs for each section. Resumable in the same way as the main-table notebook.

## 1. Mount Drive and set up paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ARPG-assets')

# ---- Section A: wall-clock ------------------------------------------------
WALLCLOCK_ROOT = DRIVE_ROOT / 'results' / 'final-paper' / 'wallclock-rho07'
WALLCLOCK_ROOT.mkdir(parents=True, exist_ok=True)
WALLCLOCK_CSV = WALLCLOCK_ROOT / 'timing.csv'
WALLCLOCK_SUMMARY_JSON = WALLCLOCK_ROOT / 'timing_summary.json'
WALLCLOCK_LOG_DIR = WALLCLOCK_ROOT / 'logs'
WALLCLOCK_LOG_DIR.mkdir(parents=True, exist_ok=True)

# ---- Section B: selection-rule ablation ----------------------------------
ABLATION_ROOT = DRIVE_ROOT / 'results' / 'final-paper' / 'selection-rule-ablation'
ABLATION_ROOT.mkdir(parents=True, exist_ok=True)
ABLATION_CSV = ABLATION_ROOT / 'results.csv'
ABLATION_LOG_DIR = ABLATION_ROOT / 'logs'
ABLATION_REJECTION_LOG_DIR = ABLATION_ROOT / 'rejection-logs'
ABLATION_LOG_DIR.mkdir(parents=True, exist_ok=True)
ABLATION_REJECTION_LOG_DIR.mkdir(parents=True, exist_ok=True)

# Main-table CSV: source of random-at-rho07 rows for the ablation
MAIN_TABLE_CSV = DRIVE_ROOT / 'results' / 'final-paper' / 'main-table' / 'results.csv'

# ---- Shared ---------------------------------------------------------------
REPO_LOCAL = Path('/content/ARPG-main')
LOCAL_SAMPLE_DIR = Path('/content/samples')
LOCAL_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

REF_NPZ = DRIVE_ROOT / 'eval' / 'VIRTUAL_imagenet256_labeled.npz'
ARPG_CKPT = DRIVE_ROOT / 'weights' / 'arpg_300m.pt'
VQ_CKPT = DRIVE_ROOT / 'weights' / 'vq_ds16_c2i.pt'
GD_REPO = DRIVE_ROOT / 'external' / 'guided-diffusion'

print(f'Wall-clock root: {WALLCLOCK_ROOT}')
print(f'Ablation root  : {ABLATION_ROOT}')
print(f'Main-table CSV : {MAIN_TABLE_CSV} (exists: {MAIN_TABLE_CSV.exists()})')

## 2. Clone repo and verify assets

In [ ]:
REPO_URL = 'https://github.com/rshahbazov23/comp447-arpg-private.git'
GITHUB_TOKEN = None  # e.g. 'ghp_...'

import subprocess, shutil

def _clone_url(url, token):
    if token and url.startswith('https://github.com/'):
        return url.replace('https://', f'https://{token}@')
    return url

if not REPO_LOCAL.exists():
    print(f'Cloning {REPO_URL}')
    subprocess.run(['git', 'clone', _clone_url(REPO_URL, GITHUB_TOKEN), str(REPO_LOCAL)], check=True)
else:
    print('Repo present, pulling')
    subprocess.run(['git', '-C', str(REPO_LOCAL), 'pull'], check=True)

for p, name in [
    (REF_NPZ,   'ImageNet reference batch'),
    (ARPG_CKPT, 'ARPG-L checkpoint'),
    (VQ_CKPT,   'VQ tokenizer'),
    (GD_REPO / 'evaluations' / 'evaluator.py', 'guided-diffusion evaluator'),
    (REPO_LOCAL / 'sample_c2i_ddp.py', 'ARPG sampling script'),
    (REPO_LOCAL / 'models' / 'confidence.py', 'Confidence module'),
]:
    if not p.exists():
        raise FileNotFoundError(f'MISSING: {name} → {p}')

conf_src = (REPO_LOCAL / 'models' / 'confidence.py').read_text()
if 'random_score' not in conf_src:
    raise RuntimeError('Random support missing — push 605038a+ from local Mac.')
print('All assets present, random support confirmed.')

## 3. Install dependencies

In [ ]:
subprocess.run(['pip', 'install', '-q',
    'einops', 'transformers', 'scipy', 'tensorflow', 'pandas',
], check=True)

import torch, pandas as pd
print(f'torch: {torch.__version__}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
assert torch.cuda.is_available(), 'No CUDA — switch the runtime to GPU.'

## 4. Shared helper functions

In [ ]:
import re, time, json, traceback
from datetime import datetime

RTR_CAP = 0.7
RTR_THRESHOLD_RANDOM = 2.0  # unreachable for random
RTR_THRESHOLD_CONF   = 0.5  # standard tau for confidence variants (empirically dead, cap binds)


def base_folder_name(step, seed, suffix_tag=''):
    s = (
        f'ARPG-L-arpg_300m-size-256-size-256-VQ-16-'
        f'topk-0-topp-1.0-temperature-1.0-cfg-5.0-cfg-schedule-linear-'
        f'sample-schedule-arccos-step-{step}-seed-{seed}'
    )
    if suffix_tag:
        s += suffix_tag
    return s


def cleanup_local_samples():
    if LOCAL_SAMPLE_DIR.exists():
        shutil.rmtree(LOCAL_SAMPLE_DIR)
    LOCAL_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)


def run_sampling_cmd(cmd, log_handle=None, prefix=''):
    """Run a torchrun sampling command, stream output with heartbeats, return elapsed seconds."""
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=str(REPO_LOCAL),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    last_print = t0
    for line in proc.stdout:
        if log_handle is not None:
            log_handle.write(line); log_handle.flush()
        now = time.time()
        if now - last_print > 60:
            print(f'{prefix}  [{(now-t0)/60:.1f} min] {line.rstrip()[:120]}')
            last_print = now
    proc.wait()
    elapsed = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f'Sampling failed: exit {proc.returncode}')
    return elapsed


_METRIC_LINE = re.compile(r'^\s*(FID|sFID|Inception Score|Precision|Recall)\s*:\s*([0-9.eE+\-]+)')

def evaluate_fid(local_npz, log_handle=None):
    cmd = ['python', 'evaluations/evaluator.py', str(REF_NPZ), str(local_npz)]
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=str(GD_REPO), capture_output=True, text=True)
    print(f'  FID eval: {(time.time()-t0)/60:.1f} min (exit {proc.returncode})')
    if log_handle is not None:
        log_handle.write('--- evaluator stdout ---\n')
        log_handle.write(proc.stdout)
        log_handle.write('\n--- evaluator stderr ---\n')
        log_handle.write(proc.stderr)
        log_handle.flush()
    if proc.returncode != 0:
        print('STDOUT (tail):', proc.stdout[-2000:])
        print('STDERR (tail):', proc.stderr[-2000:])
        raise RuntimeError(f'FID eval failed: exit {proc.returncode}')
    metrics = {}
    for line in proc.stdout.splitlines():
        m = _METRIC_LINE.match(line)
        if m:
            key = m.group(1).lower().replace(' ', '_')
            metrics[key] = float(m.group(2))
    if 'fid' not in metrics:
        print('STDOUT:', proc.stdout)
        raise ValueError('Could not parse FID')
    return metrics


print('Shared helpers loaded.')

---

## Section A — E1: Wall-clock at ρ=0.7

**Configs.** Vanilla + RTR(random, ρ=0.7) × steps {8, 16, 32} × 5 reps = 30 timed runs.
**Samples per run.** 2,000 (timing-only; no FID eval).
**Output.** `timing.csv` (one row per rep) + `timing_summary.json` (mean ± std per config).
**Time.** ~1.5 hours.

### A.1 — Config matrix and timing CSV

In [ ]:
WALLCLOCK_STEPS = [8, 16, 32]
WALLCLOCK_REPS = 5
WALLCLOCK_NUM_SAMPLES = 2000
WALLCLOCK_BATCH = 64

def make_wallclock_configs():
    out = []
    for mode in ['vanilla', 'rtr']:
        for step in WALLCLOCK_STEPS:
            for rep in range(WALLCLOCK_REPS):
                out.append({'mode': mode, 'step': step, 'rep': rep})
    return out

WALLCLOCK_CONFIGS = make_wallclock_configs()
print(f'Wall-clock configs: {len(WALLCLOCK_CONFIGS)} (2 modes × 3 steps × 5 reps)')

if not WALLCLOCK_CSV.exists():
    pd.DataFrame(columns=[
        'mode', 'step', 'rep', 'seconds', 'num_samples', 'cap', 'timestamp',
    ]).to_csv(WALLCLOCK_CSV, index=False)
    print(f'Initialised empty timing CSV: {WALLCLOCK_CSV}')
else:
    print(f'Timing CSV exists with {len(pd.read_csv(WALLCLOCK_CSV))} rows.')

df_wc = pd.read_csv(WALLCLOCK_CSV)
done_wc = set((r['mode'], int(r['step']), int(r['rep']))
              for _, r in df_wc.iterrows()) if len(df_wc) else set()
remaining_wc = [c for c in WALLCLOCK_CONFIGS
                if (c['mode'], c['step'], c['rep']) not in done_wc]
print(f'\nProgress: {len(WALLCLOCK_CONFIGS) - len(remaining_wc)}/{len(WALLCLOCK_CONFIGS)} done, {len(remaining_wc)} remaining')

### A.2 — Wall-clock main loop

In [ ]:
started_A = datetime.now()
fails_A = []

for i, cfg in enumerate(WALLCLOCK_CONFIGS, 1):
    print(f'\n[A {i}/{len(WALLCLOCK_CONFIGS)}] mode={cfg["mode"]} step={cfg["step"]} rep={cfg["rep"]}')

    df_wc = pd.read_csv(WALLCLOCK_CSV)
    if len(df_wc) > 0:
        mask = (
            (df_wc['mode'] == cfg['mode'])
            & (df_wc['step'].astype(int) == cfg['step'])
            & (df_wc['rep'].astype(int) == cfg['rep'])
        )
        if mask.any():
            print('  SKIP (already timed)')
            continue

    # Use a unique seed per (rep) so reps differ; but cap timing isn't seed-sensitive in expectation.
    seed = cfg['rep']

    folder = base_folder_name(cfg['step'], seed,
                              suffix_tag=(f'-mode-rejection-metric-random-tau-2.0-cap-0.7' if cfg['mode'] == 'rtr' else ''))
    log_path = WALLCLOCK_LOG_DIR / f'{cfg["mode"]}-step{cfg["step"]}-rep{cfg["rep"]}.log'

    cmd = [
        'torchrun', '--nnodes=1', '--nproc_per_node=1',
        'sample_c2i_ddp.py',
        '--gpt-model', 'ARPG-L',
        '--gpt-ckpt', str(ARPG_CKPT),
        '--vq-ckpt', str(VQ_CKPT),
        '--sample-schedule', 'arccos',
        '--cfg-schedule', 'linear',
        '--cfg-scale', '5.0',
        '--step', str(cfg['step']),
        '--per-proc-batch-size', str(WALLCLOCK_BATCH),
        '--num-fid-samples', str(WALLCLOCK_NUM_SAMPLES),
        '--global-seed', str(seed),
        '--sample-dir', str(LOCAL_SAMPLE_DIR),
        '--no-compile',
        '--precision', 'bf16',
    ]
    if cfg['mode'] == 'rtr':
        cmd += [
            '--rejection-mode', 'rejection',
            '--confidence-metric', 'random',
            '--rejection-threshold', str(RTR_THRESHOLD_RANDOM),
            '--max-reject-rate', str(RTR_CAP),
        ]

    try:
        with open(log_path, 'w') as log_h:
            seconds = run_sampling_cmd(cmd, log_handle=log_h, prefix='   ')

        row = {
            'mode': cfg['mode'],
            'step': cfg['step'],
            'rep': cfg['rep'],
            'seconds': round(seconds, 3),
            'num_samples': WALLCLOCK_NUM_SAMPLES,
            'cap': RTR_CAP if cfg['mode'] == 'rtr' else None,
            'timestamp': datetime.now().isoformat(),
        }
        df_wc = pd.read_csv(WALLCLOCK_CSV)
        df_wc = pd.concat([df_wc, pd.DataFrame([row])], ignore_index=True)
        df_wc.to_csv(WALLCLOCK_CSV, index=False)
        print(f'  DONE   {seconds:.1f} s')
    except Exception as e:
        print(f'  FAILED: {e}')
        traceback.print_exc()
        fails_A.append({'config': cfg, 'error': str(e)})
    finally:
        cleanup_local_samples()

    elapsed_h = (datetime.now() - started_A).total_seconds() / 3600
    print(f'  Section A elapsed: {elapsed_h:.2f} h')

print(f'\n=== Section A complete. Failures: {len(fails_A)} ===')

### A.3 — Wall-clock summary

In [ ]:
import numpy as np

df = pd.read_csv(WALLCLOCK_CSV)
if len(df) == 0:
    raise RuntimeError('Timing CSV is empty — run section A.2 first.')
df['step'] = df['step'].astype(int)

summary = (
    df.groupby(['mode', 'step'])['seconds']
      .agg(['min', 'mean', 'std', 'count'])
      .round(3)
      .sort_index()
)
print(f'\nWall-clock (2K samples per rep, cap ρ=0.7 for RTR)\n')
print(summary.to_string())

# Overhead per step count
print(f'\n\nRTR overhead vs vanilla (mean per-rep seconds):')
summary_out = {}
for step in sorted(df['step'].unique()):
    v = df[(df['mode']=='vanilla') & (df['step']==step)]['seconds'].mean()
    r = df[(df['mode']=='rtr')     & (df['step']==step)]['seconds'].mean()
    if not (np.isnan(v) or np.isnan(r)):
        ratio = r / v
        pct = 100 * (ratio - 1)
        print(f'  {step:>3} steps:   vanilla={v:.2f}s   RTR={r:.2f}s   ratio={ratio:.3f}   overhead={pct:+.2f}%')
        summary_out[str(step)] = {
            'vanilla_mean_s': round(v, 3),
            'rtr_mean_s': round(r, 3),
            'ratio': round(ratio, 4),
            'overhead_pct': round(pct, 3),
        }

# Persist summary as JSON for the paper
with open(WALLCLOCK_SUMMARY_JSON, 'w') as f:
    json.dump({
        'cap_rho': RTR_CAP,
        'num_samples_per_rep': WALLCLOCK_NUM_SAMPLES,
        'batch_size': WALLCLOCK_BATCH,
        'reps_per_config': WALLCLOCK_REPS,
        'per_step': summary_out,
    }, f, indent=2)
print(f'\nSummary written to {WALLCLOCK_SUMMARY_JSON}')

---

## Section B — E2: Selection-rule ablation at FID-50K

**Configs.** 4 selection rules × {8, 16 steps} × 3 seeds = 24 cells. Random rows (6) pre-load from the main-table CSV — only 18 new FID-50K runs.

| Selection rule | τ |
|---|---|
| random   | 2.0 (unreachable) |
| margin   | 0.5 |
| max_prob | 0.5 |
| entropy  | 0.5 |

All at ρ=0.7.

**Time.** ~12 hours for the 18 new FID-50K runs.

### B.1 — Config matrix + pre-load random rows from main-table

In [ ]:
ABLATION_METRICS = ['random', 'margin', 'max_prob', 'entropy']
ABLATION_STEPS = [16, 8]   # 16 first (headline regime), 8 second
ABLATION_SEEDS = [0, 1, 2]
ABLATION_NUM_SAMPLES = 50_000
ABLATION_BATCH = 64

def make_ablation_configs():
    out = []
    for step in ABLATION_STEPS:
        for metric in ABLATION_METRICS:
            for seed in ABLATION_SEEDS:
                out.append({'metric': metric, 'step': step, 'seed': seed})
    return out

ABLATION_CONFIGS = make_ablation_configs()
print(f'Ablation configs: {len(ABLATION_CONFIGS)} (2 steps × 4 metrics × 3 seeds)')

# Initialise CSV
if not ABLATION_CSV.exists():
    pd.DataFrame(columns=[
        'metric', 'step', 'seed', 'tau', 'cap',
        'fid', 'inception_score', 'sfid', 'precision', 'recall',
        'source', 'timestamp',
    ]).to_csv(ABLATION_CSV, index=False)
    print(f'Initialised empty ablation CSV: {ABLATION_CSV}')

# Pre-load random rows from the main-table CSV (rtr mode at rho=0.7).
# Main-table rows have columns: mode, step, seed, fid, inception_score, sfid, precision, recall.
# 'mode' == 'rtr' there corresponds to (metric='random', tau=2.0, cap=0.7) here.
if MAIN_TABLE_CSV.exists():
    df_main = pd.read_csv(MAIN_TABLE_CSV)
    df_ab = pd.read_csv(ABLATION_CSV)
    existing_keys = set((str(r['metric']), int(r['step']), int(r['seed']))
                        for _, r in df_ab.iterrows()) if len(df_ab) else set()

    new_rows = []
    for step in ABLATION_STEPS:
        for seed in ABLATION_SEEDS:
            if ('random', step, seed) in existing_keys:
                continue
            row = df_main[(df_main['mode'] == 'rtr')
                          & (df_main['step'].astype(int) == step)
                          & (df_main['seed'].astype(int) == seed)]
            if len(row) == 0:
                print(f'  NOTE: no main-table row for random/step={step}/seed={seed}; will sample fresh')
                continue
            r = row.iloc[0]
            new_rows.append({
                'metric': 'random',
                'step': step,
                'seed': seed,
                'tau': RTR_THRESHOLD_RANDOM,
                'cap': RTR_CAP,
                'fid': float(r['fid']),
                'inception_score': float(r['inception_score']) if pd.notna(r.get('inception_score')) else None,
                'sfid': float(r['sfid']) if pd.notna(r.get('sfid')) else None,
                'precision': float(r['precision']) if pd.notna(r.get('precision')) else None,
                'recall': float(r['recall']) if pd.notna(r.get('recall')) else None,
                'source': 'main-table-preload',
                'timestamp': datetime.now().isoformat(),
            })
    if new_rows:
        df_ab = pd.concat([df_ab, pd.DataFrame(new_rows)], ignore_index=True)
        df_ab.to_csv(ABLATION_CSV, index=False)
        print(f'Pre-loaded {len(new_rows)} random rows from main-table CSV.')
else:
    print(f'Main-table CSV not found at {MAIN_TABLE_CSV} — random rows will be sampled fresh.')

df_ab = pd.read_csv(ABLATION_CSV)
done_ab = set((str(r['metric']), int(r['step']), int(r['seed']))
              for _, r in df_ab.iterrows()) if len(df_ab) else set()
remaining_ab = [c for c in ABLATION_CONFIGS
                if (c['metric'], c['step'], c['seed']) not in done_ab]
print(f'\nAblation progress: {len(ABLATION_CONFIGS) - len(remaining_ab)}/{len(ABLATION_CONFIGS)} done, {len(remaining_ab)} remaining')
if remaining_ab[:6]:
    print('Next 6 configs:')
    for c in remaining_ab[:6]:
        print(f'  metric={c["metric"]:<9} step={c["step"]:>2} seed={c["seed"]}')

### B.2 — Selection-rule ablation main loop

In [ ]:
def ablation_folder(metric, step, seed):
    tau = RTR_THRESHOLD_RANDOM if metric == 'random' else RTR_THRESHOLD_CONF
    suffix = f'-mode-rejection-metric-{metric}-tau-{tau}-cap-{RTR_CAP}'
    return base_folder_name(step, seed, suffix_tag=suffix)


def run_ablation_config(cfg, log_handle=None):
    metric = cfg['metric']
    step = cfg['step']
    seed = cfg['seed']
    tau = RTR_THRESHOLD_RANDOM if metric == 'random' else RTR_THRESHOLD_CONF
    folder = ablation_folder(metric, step, seed)

    cmd = [
        'torchrun', '--nnodes=1', '--nproc_per_node=1',
        'sample_c2i_ddp.py',
        '--gpt-model', 'ARPG-L',
        '--gpt-ckpt', str(ARPG_CKPT),
        '--vq-ckpt', str(VQ_CKPT),
        '--sample-schedule', 'arccos',
        '--cfg-schedule', 'linear',
        '--cfg-scale', '5.0',
        '--step', str(step),
        '--per-proc-batch-size', str(ABLATION_BATCH),
        '--num-fid-samples', str(ABLATION_NUM_SAMPLES),
        '--global-seed', str(seed),
        '--sample-dir', str(LOCAL_SAMPLE_DIR),
        '--no-compile',
        '--precision', 'bf16',
        '--rejection-mode', 'rejection',
        '--confidence-metric', metric,
        '--rejection-threshold', str(tau),
        '--max-reject-rate', str(RTR_CAP),
        '--log-json', str(ABLATION_REJECTION_LOG_DIR / f'{folder}.json'),
    ]
    print(f'  Sampling: metric={metric} step={step} seed={seed} tau={tau}')
    elapsed = run_sampling_cmd(cmd, log_handle=log_handle, prefix='   ')
    print(f'  Sampling done in {elapsed/60:.1f} min')

    local_npz = LOCAL_SAMPLE_DIR / f'{folder}.npz'
    if not local_npz.exists():
        raise FileNotFoundError(f'NPZ not produced: {local_npz}')
    return local_npz


def append_ablation_row(cfg, metrics):
    df = pd.read_csv(ABLATION_CSV)
    row = {
        'metric': cfg['metric'],
        'step': cfg['step'],
        'seed': cfg['seed'],
        'tau': RTR_THRESHOLD_RANDOM if cfg['metric'] == 'random' else RTR_THRESHOLD_CONF,
        'cap': RTR_CAP,
        'fid': metrics.get('fid'),
        'inception_score': metrics.get('inception_score'),
        'sfid': metrics.get('sfid'),
        'precision': metrics.get('precision'),
        'recall': metrics.get('recall'),
        'source': 'ablation-run',
        'timestamp': datetime.now().isoformat(),
    }
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(ABLATION_CSV, index=False)
    return df


started_B = datetime.now()
fails_B = []

for i, cfg in enumerate(ABLATION_CONFIGS, 1):
    print(f'\n[B {i}/{len(ABLATION_CONFIGS)}] metric={cfg["metric"]} step={cfg["step"]} seed={cfg["seed"]}')

    df_ab = pd.read_csv(ABLATION_CSV)
    key = (cfg['metric'], cfg['step'], cfg['seed'])
    done_set = set((str(r['metric']), int(r['step']), int(r['seed']))
                   for _, r in df_ab.iterrows()) if len(df_ab) else set()
    if key in done_set:
        print('  SKIP (already in CSV — may be a pre-loaded random row)')
        continue

    folder = ablation_folder(cfg['metric'], cfg['step'], cfg['seed'])
    log_path = ABLATION_LOG_DIR / f'{folder}.log'
    local_npz = LOCAL_SAMPLE_DIR / f'{folder}.npz'

    try:
        with open(log_path, 'w') as log_h:
            local_npz = run_ablation_config(cfg, log_handle=log_h)
            metrics = evaluate_fid(local_npz, log_handle=log_h)
            append_ablation_row(cfg, metrics)
        print(f'  DONE  FID={metrics["fid"]:.4f}  IS={metrics.get("inception_score", 0):.2f}  Prec={metrics.get("precision", 0):.3f}  Rec={metrics.get("recall", 0):.3f}')
    except Exception as e:
        print(f'  FAILED: {e}')
        traceback.print_exc()
        fails_B.append({'config': cfg, 'error': str(e), 'log': str(log_path)})
    finally:
        cleanup_local_samples()

    elapsed_h = (datetime.now() - started_B).total_seconds() / 3600
    print(f'  Section B elapsed: {elapsed_h:.2f} h')

print(f'\n=== Section B complete. Failures: {len(fails_B)} ===')
for f in fails_B:
    print(f'  {f["config"]} → {f["error"]}')

### B.3 — Selection-rule summary

In [ ]:
df = pd.read_csv(ABLATION_CSV)
if len(df) == 0:
    raise RuntimeError('Ablation CSV is empty.')
df['step'] = df['step'].astype(int)
df['seed'] = df['seed'].astype(int)

summary = (
    df.groupby(['step', 'metric'])['fid']
      .agg(['mean', 'std', 'count'])
      .round(4)
      .sort_index()
)
print('FID-50K — selection-rule comparison (mean ± std, n)\n')
print(summary.to_string())

# Pivot for §4.2 ablation table view
print('\n\n§4.2 ablation table (rows=metric, cols=step):')
pivot = (
    df.groupby(['step', 'metric'])['fid'].mean().unstack('step').round(4)
)
# Order metrics so random is first (the headline / control)
metric_order = ['random', 'margin', 'max_prob', 'entropy']
pivot = pivot.reindex([m for m in metric_order if m in pivot.index])
print(pivot.to_string())

# Deltas vs random control
print('\n\nDeltas vs random (mean FID-50K):')
for step in sorted(df['step'].unique()):
    rand_fid = df[(df['step']==step) & (df['metric']=='random')]['fid'].mean()
    if np.isnan(rand_fid):
        print(f'  {step:>3} steps:   random row missing — cannot compute deltas')
        continue
    print(f'  {step:>3} steps:   random = {rand_fid:.4f}')
    for metric in ['margin', 'max_prob', 'entropy']:
        m_fid = df[(df['step']==step) & (df['metric']==metric)]['fid'].mean()
        if not np.isnan(m_fid):
            delta = m_fid - rand_fid
            print(f'              {metric:<8} = {m_fid:.4f}   Δ vs random = {delta:+.4f}')

# Save summary CSV
summary_path = ABLATION_ROOT / 'summary.csv'
summary.to_csv(summary_path)
print(f'\nWrote summary CSV to {summary_path}')